# dense와 sparse 비교

이 노트북은 빈 Python 3.12 커널에서 순서대로 실행합니다.


## 실행 환경 준비

같은 12개 문서에 OpenAI dense 검색과 Kiwi+BM25 sparse 검색을 적용합니다. API 예상 비용은 미화 0.01달러 미만입니다.


In [1]:
%pip install -q "numpy==2.5.3" "kiwipiepy==0.23.2" "rank-bm25==0.2.2" "python-dotenv==1.2.3" "langchain-openai==1.6.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os
from pathlib import Path
import numpy as np
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings

# 현재 폴더부터 상위 폴더에서 .env를 찾고 키의 존재만 확인합니다.
env_file = find_dotenv(usecwd=True)
assert env_file, "프로젝트 루트에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."
course_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "labs/part3/ch12-retrieval-tuning/data").is_dir()), None)
assert course_root is not None, "Ch 12 실습 데이터 폴더를 찾지 못했습니다."
chapter_dir = course_root / "labs/part3/ch12-retrieval-tuning"
data_dir = chapter_dir / "data"
docs = json.loads((data_dir / "core-documents.json").read_text())
queries = json.loads((data_dir / "core-queries.json").read_text())
embedder = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)

In [3]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

In [4]:
texts = [f"{doc['title']} {doc['text']}" for doc in docs]
doc_vectors = np.array(embedder.embed_documents(texts))
query_vectors = np.array([embedder.embed_query(q["question"]) for q in queries])
kiwi = Kiwi(); keep = ("N", "V", "VA", "MAG", "SN", "SL")
def terms(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith(keep)]
bm25 = BM25Okapi([terms(text) for text in texts])
def dense_rank(vector, k=3):
    scores = doc_vectors @ vector / (np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(vector))
    return [docs[i]["doc_id"] for i in np.argsort(-scores)[:k]]
def sparse_rank(question, k=3):
    scores = bm25.get_scores(terms(question))
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:k]
    return [docs[i]["doc_id"] for i in order]

## 기본 실행

의미를 바꾸어 표현한 질문에서 두 검색기의 상위 결과를 확인합니다.


In [5]:
query = queries[0]
print("질문:", query["question"], "정답:", query["gold_doc_id"])
print("dense:", dense_rank(query_vectors[0]))
print("sparse:", sparse_rank(query["question"]))

질문: 책을 빌릴 수 있는 기간은 며칠인가요? 정답: LIB-001
dense: ['LIB-004', 'LIB-001', 'LIB-003']
sparse: ['LIB-004', 'LIB-003', 'LIB-001']


## 중간 결과 관찰

ISBN처럼 정확한 표기가 있는 질문도 같은 방법으로 비교합니다.


In [6]:
query = queries[2]
print("질문:", query["question"], "정답:", query["gold_doc_id"])
print("dense:", dense_rank(query_vectors[2]))
print("sparse:", sparse_rank(query["question"]))

질문: ISBN으로 없는 책을 신청하려면? 정답: LIB-003
dense: ['LIB-003', 'ACC-002', 'LIB-004']
sparse: ['LIB-003', 'ACC-003', 'LIB-004']


## 값 하나를 바꿔 비교

검색기만 바꾸고 질문·문서·k=3을 유지합니다.


In [7]:
for query, vector in zip(queries, query_vectors):
    dense_hit = query["gold_doc_id"] in dense_rank(vector)
    sparse_hit = query["gold_doc_id"] in sparse_rank(query["question"])
    print(query["qid"], query["kind"], "dense", dense_hit, "sparse", sparse_hit)

C01 semantic dense True sparse True
C02 semantic dense False sparse True
C03 keyword dense True sparse True
C04 keyword dense True sparse True
C05 semantic dense True sparse True
C06 semantic dense True sparse True
C07 keyword dense True sparse True
C08 semantic dense True sparse True


## 반복 실험과 결과 해석

검색기별 Recall@3과 성공 여부가 달랐던 질문을 저장합니다.


In [8]:
rows = []
for query, vector in zip(queries, query_vectors):
    dense = dense_rank(vector); sparse = sparse_rank(query["question"])
    rows.append({"qid": query["qid"], "kind": query["kind"], "dense": dense,
                 "sparse": sparse, "dense_hit": query["gold_doc_id"] in dense,
                 "sparse_hit": query["gold_doc_id"] in sparse})
for name in ["dense", "sparse"]:
    print(name, "Recall@3:", sum(row[f"{name}_hit"] for row in rows), "/", len(rows))
print("성공 여부가 다른 질문:", [r["qid"] for r in rows if r["dense_hit"] != r["sparse_hit"]])
print("기록할 결과 행:", len(rows))

dense Recall@3: 7 / 8
sparse Recall@3: 8 / 8
성공 여부가 다른 질문: ['C02']
기록할 결과 행: 8


<h3>실행 결과 보존</h3><p>기존 정식 결과를 해시 이름으로 보존하고, 이번 결과를 실행 시각이 포함된 파일에 먼저 저장한 뒤 최신 결과 파일을 갱신합니다.</p>

In [9]:
import hashlib
from datetime import UTC, datetime
payload = json.dumps(rows, ensure_ascii=False, indent=2)
runs = chapter_dir / "runs"; history = runs / "core-history"
history.mkdir(parents=True, exist_ok=True)
pointer = runs / "core-05-dense-sparse.json"
if pointer.exists():
    old = pointer.read_bytes()
    backup = history / f"before-{hashlib.sha256(old).hexdigest()[:12]}-{pointer.name}"
    if not backup.exists(): backup.write_bytes(old)
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
archive = history / f"{run_id}-{pointer.name}"
archive.write_text(payload, encoding="utf-8")
pointer.write_text(payload, encoding="utf-8")
print("실행별 보존 로그:", archive.relative_to(course_root))

실행별 보존 로그: labs/part3/ch12-retrieval-tuning/runs/core-history/20260909T174515609696-core-05-dense-sparse.json
